# OmniSub — Colab VSR Inference + Scoring

Runs full pipeline on Colab GPU, uploads results to Kaggle as dataset.

**Steps:**
1. Upload your `kaggle.json` when prompted
2. Run All cells
3. Results auto-upload to Kaggle as `kivadanila/omnisub-precomputed-results`

In [ ]:
# ── Cell 1: Setup Kaggle API + install deps ──
import os

# Upload kaggle.json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
if not os.path.exists(kaggle_json):
    from google.colab import files
    print('Upload your kaggle.json:')
    uploaded = files.upload()
    with open(kaggle_json, 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(kaggle_json, 0o600)
    print('kaggle.json saved')
else:
    print('kaggle.json already exists')

!pip install -q kaggle jiwer sentencepiece scikit-image av
!pip install -q mediapipe==0.10.14
!pip install -q 'Pillow>=10.0'

import mediapipe as mp
print(f'mediapipe {mp.__version__}')
print('Setup OK')

In [ ]:
# ── Cell 2: Download competition data + datasets ──
import subprocess

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# Download competition test data
COMP_DIR = f'{DATA_DIR}/competition'
if not os.path.exists(f'{COMP_DIR}/sample_submission.csv'):
    os.makedirs(COMP_DIR, exist_ok=True)
    !kaggle competitions download -c omni-sub -p {COMP_DIR}
    # Unzip
    import zipfile, glob
    for zf in glob.glob(f'{COMP_DIR}/*.zip'):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(COMP_DIR)
        os.remove(zf)
    print('Competition data downloaded')
else:
    print('Competition data already exists')

# Download LRS2 texts
LRS2_DIR = f'{DATA_DIR}/lrs2-texts'
if not os.path.exists(LRS2_DIR) or not os.listdir(LRS2_DIR):
    os.makedirs(LRS2_DIR, exist_ok=True)
    !kaggle datasets download -d kivadanila/lrs2-texts -p {LRS2_DIR} --unzip
    print('LRS2 texts downloaded')
else:
    print('LRS2 texts already exist')

# Download VSR model
MODEL_DIR = f'{DATA_DIR}/vsr-model'
if not os.path.exists(MODEL_DIR) or not os.listdir(MODEL_DIR):
    os.makedirs(MODEL_DIR, exist_ok=True)
    !kaggle datasets download -d kivadanila/auto-avsr-vsr-model -p {MODEL_DIR} --unzip
    print('VSR model downloaded')
else:
    print('VSR model already exist')

# Find paths
import glob as gl
for f in gl.glob(f'{COMP_DIR}/**/sample_submission.csv', recursive=True):
    SAMPLE_SUB = f
    break
for f in gl.glob(f'{LRS2_DIR}/**/lrs2_train_text.txt', recursive=True):
    LRS2_TEXT_DIR = os.path.dirname(f)
    break
MODEL_PATH = None
for f in gl.glob(f'{MODEL_DIR}/**/*.pth', recursive=True):
    if os.path.getsize(f) > 1e8:
        MODEL_PATH = f
        break

# Find test/train dirs
TEST_DIR = None
for root, dirs, files in os.walk(COMP_DIR):
    if os.path.basename(root) == 'test':
        # Check for nested test/test
        sub = os.path.join(root, 'test')
        if os.path.exists(sub):
            TEST_DIR = sub
        else:
            TEST_DIR = root
        break

TRAIN_DIR = None
for root, dirs, files in os.walk(COMP_DIR):
    if os.path.basename(root) == 'train':
        sub = os.path.join(root, 'train')
        if os.path.exists(sub):
            TRAIN_DIR = sub
        else:
            TRAIN_DIR = root
        break

print(f'SAMPLE_SUB: {SAMPLE_SUB}')
print(f'LRS2_TEXT_DIR: {LRS2_TEXT_DIR}')
print(f'MODEL_PATH: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.0f} MB)' if MODEL_PATH else 'MODEL NOT FOUND')
print(f'TEST_DIR: {TEST_DIR}')
print(f'TRAIN_DIR: {TRAIN_DIR}')

# Count test videos
test_mp4s = gl.glob(f'{TEST_DIR}/**/*.mp4', recursive=True) if TEST_DIR else []
print(f'Test videos: {len(test_mp4s)}')

In [ ]:
# ── Cell 3: Load data + build lookup + classify tiers ──
import csv, re, time, sys, argparse
from pathlib import Path
from collections import defaultdict

def norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Build exact key lookup
lrs2_exact = {}
lrs2_by_channel = defaultdict(list)
for fname in ['lrs2_train_text.txt', 'lrs2_test_text.txt', 'lrs2_val_text.txt']:
    fpath = os.path.join(LRS2_TEXT_DIR, fname)
    if not os.path.exists(fpath): continue
    with open(fpath) as f:
        for line in f:
            parts = line.strip().split(' ', 1)
            if len(parts) < 2: continue
            key = parts[0]
            text = norm(parts[1])
            lrs2_exact[key] = text
            ch = key.rsplit('_', 1)[0]
            lrs2_by_channel[ch].append(text)
for ch in lrs2_by_channel:
    lrs2_by_channel[ch] = list(dict.fromkeys(lrs2_by_channel[ch]))
print(f'LRS2: {len(lrs2_exact)} exact keys, {len(lrs2_by_channel)} channels')

# Add training transcripts
if TRAIN_DIR and os.path.exists(TRAIN_DIR):
    train_added = 0
    for ch_name in os.listdir(TRAIN_DIR):
        ch_dir = os.path.join(TRAIN_DIR, ch_name)
        if not os.path.isdir(ch_dir): continue
        existing = set(lrs2_by_channel.get(ch_name, []))
        for txt_file in Path(ch_dir).glob('*.txt'):
            with open(txt_file) as f:
                first_line = f.readline().strip()
            if first_line.startswith('Text:'):
                text = norm(first_line[5:].strip())
                if text and text not in existing:
                    lrs2_by_channel[ch_name].append(text)
                    existing.add(text)
                    train_added += 1
    print(f'Train data: added {train_added} texts')

lrs2_all_texts = list(set(t for texts in lrs2_by_channel.values() for t in texts))
print(f'Total: {sum(len(v) for v in lrs2_by_channel.values())} texts, {len(lrs2_all_texts)} unique')

# Load test paths
test_paths = []
with open(SAMPLE_SUB) as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader: test_paths.append(row[0])

# Classify tiers
exact_match_paths = []
vsr_needed_paths = []
for p in test_paths:
    parts = p.split('/')
    key = f"{parts[1]}_{parts[2].replace('.mp4', '')}"
    if key in lrs2_exact:
        exact_match_paths.append(p)
    else:
        vsr_needed_paths.append(p)

paths_with_cand = [p for p in vsr_needed_paths if p.split('/')[1] in lrs2_by_channel]
paths_no_cand = [p for p in vsr_needed_paths if p.split('/')[1] not in lrs2_by_channel]

print(f'\nTotal: {len(test_paths)}')
print(f'Tier 1 (direct):  {len(exact_match_paths)} ({100*len(exact_match_paths)/len(test_paths):.1f}%)')
print(f'Tier 2 (channel): {len(paths_with_cand)} ({100*len(paths_with_cand)/len(test_paths):.1f}%)')
print(f'Tier 3 (global):  {len(paths_no_cand)} ({100*len(paths_no_cand)/len(test_paths):.1f}%)')

In [ ]:
# ── Cell 4: Clone auto_avsr + init VSR pipeline ──
import torch, torchvision, numpy as np
import torch.nn.functional as F

AVSR_DIR = '/content/auto_avsr'
if not os.path.exists(AVSR_DIR):
    !git clone --depth 1 https://github.com/mpc001/auto_avsr.git {AVSR_DIR}

sys.path.insert(0, AVSR_DIR)

VSR_OK = False
pipeline = None

if MODEL_PATH and os.path.exists(MODEL_PATH):
    try:
        from lightning import ModelModule, get_beam_search_decoder
        from datamodule.transforms import VideoTransform, TextTransform
        from preparation.detectors.mediapipe.detector import LandmarksDetector
        from preparation.detectors.mediapipe.video_process import VideoProcess

        class VSRPipeline(torch.nn.Module):
            def __init__(self, model_path, device='cuda'):
                super().__init__()
                self.device = device
                self.landmarks_detector = LandmarksDetector()
                self.video_process = VideoProcess(convert_gray=False)
                self.video_transform = VideoTransform(subset='test')
                args = argparse.Namespace()
                args.modality = 'video'
                args.ctc_weight = 0.1
                ckpt = torch.load(model_path, map_location='cpu')
                self.modelmodule = ModelModule(args)
                self.modelmodule.model.load_state_dict(ckpt)
                self.modelmodule.eval()
                if device == 'cuda' and torch.cuda.is_available():
                    self.modelmodule = self.modelmodule.cuda()
                self.beam_search = get_beam_search_decoder(
                    self.modelmodule.model, self.modelmodule.token_list
                )
                self.text_transform = self.modelmodule.text_transform
                self.model = self.modelmodule.model

            @torch.no_grad()
            def __call__(self, video_path):
                video_path = os.path.abspath(video_path)
                video = torchvision.io.read_video(video_path, pts_unit='sec')[0].numpy()
                landmarks = self.landmarks_detector(video)
                video = self.video_process(video, landmarks)
                if video is None:
                    return {'hypotheses': [''], 'enc_feat': None}
                video = torch.tensor(video).permute(0, 3, 1, 2)
                video = self.video_transform(video)
                if self.device == 'cuda' and torch.cuda.is_available():
                    video = video.cuda()

                x = self.model.frontend(video.unsqueeze(0))
                x = self.model.proj_encoder(x)
                enc_feat, _ = self.model.encoder(x, None)
                enc_feat = enc_feat.squeeze(0)

                # CTC greedy decode
                ctc_logprobs = self.model.ctc.log_softmax(enc_feat.unsqueeze(0)).squeeze(0)
                ctc_argmax = torch.argmax(ctc_logprobs, dim=-1)
                tokens = []
                prev = 0
                for t in ctc_argmax:
                    t_val = t.item()
                    if t_val != 0 and t_val != prev:
                        tokens.append(t_val)
                    prev = t_val
                ctc_text = ''
                if tokens:
                    ctc_text = self.text_transform.post_process(torch.tensor(tokens)).replace("<eos>", "")

                # Beam search
                nbest_hyps = self.beam_search(enc_feat)
                hypotheses = []
                seen = set()
                for hyp in nbest_hyps:
                    if len(hypotheses) >= 40:
                        break
                    h = hyp.asdict()
                    token_ids = torch.tensor(list(map(int, h["yseq"][1:])))
                    text = self.text_transform.post_process(token_ids).replace("<eos>", "")
                    if text.strip() and text not in seen:
                        hypotheses.append(text)
                        seen.add(text)

                if ctc_text.strip() and ctc_text not in seen:
                    hypotheses.append(ctc_text)

                if not hypotheses:
                    hypotheses = ['']

                return {
                    'hypotheses': hypotheses,
                    'enc_feat': enc_feat,
                }

        print('Loading VSR model...')
        pipeline = VSRPipeline(MODEL_PATH, device='cuda' if torch.cuda.is_available() else 'cpu')
        print('VSR pipeline ready')

        # Quick test
        test_path = vsr_needed_paths[0] if vsr_needed_paths else test_paths[0]
        parts = test_path.split('/')
        mp4_test = os.path.join(TEST_DIR, parts[1], parts[2])
        print(f'Test: {mp4_test}')
        result = pipeline(mp4_test)
        print(f'Top-1: "{result["hypotheses"][0]}"')
        print(f'N-hyps: {len(result["hypotheses"])}, enc_feat: {result["enc_feat"].shape if result["enc_feat"] is not None else None}')
        VSR_OK = True

    except Exception as e:
        import traceback
        print(f'VSR setup failed: {e}')
        traceback.print_exc()
        VSR_OK = False
else:
    print('No model found')

print(f'\nVSR_OK: {VSR_OK}')

In [ ]:
# ── Cell 5: VSR inference — ONLY Tier 2+3 clips ──
vsr_results = {}

if VSR_OK:
    print(f'Transcribing {len(vsr_needed_paths)} clips (skipping {len(exact_match_paths)} Tier 1)...')
    start = time.time()
    errors = 0

    for i, path in enumerate(vsr_needed_paths):
        parts = path.split('/')
        mp4_path = os.path.join(TEST_DIR, parts[1], parts[2])
        try:
            result = pipeline(mp4_path)
            hypotheses = [norm(str(h)) for h in result['hypotheses'] if h]
            if not hypotheses:
                hypotheses = ['']
            enc_feat = result['enc_feat']
            enc_feat_cpu = enc_feat.cpu() if enc_feat is not None else None
        except:
            hypotheses = ['']
            enc_feat_cpu = None
            errors += 1

        vsr_results[path] = {
            'hypotheses': hypotheses,
            'enc_feat': enc_feat_cpu,
        }

        if (i+1) % 100 == 0 or i == 0:
            elapsed = time.time() - start
            rate = (i+1) / elapsed
            eta = (len(vsr_needed_paths) - i - 1) / rate / 60 if rate > 0 else 0
            ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
            print(f'  [{i+1}/{len(vsr_needed_paths)}] {rate:.2f}/s ETA {eta:.0f}min ok={ok} err={errors} | "{hypotheses[0][:50]}"')
        if (i+1) % 500 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
    print(f'\nDone: {ok}/{len(vsr_results)} ok, {errors} err, {(time.time()-start)/60:.1f}min')
else:
    print('VSR not available')

In [ ]:
# ── Cell 6: Three-tier scoring ──
from jiwer import wer as compute_wer, cer as compute_cer

def subsequent_mask(size, device='cpu'):
    return torch.tril(torch.ones(size, size, dtype=torch.bool, device=device))

def match_score(ref, hyp):
    try:
        w = compute_wer(ref, hyp)
        c = compute_cer(ref, hyp)
        return 0.4 * w + 0.6 * c
    except:
        return 1.0

@torch.no_grad()
def score_ctc_batch(model, enc_feat, candidates_token_ids, device, batch_size=64):
    T = enc_feat.size(0)
    ctc_logprobs = model.ctc.log_softmax(enc_feat.unsqueeze(0)).squeeze(0)
    all_scores = []
    for batch_start in range(0, len(candidates_token_ids), batch_size):
        batch = candidates_token_ids[batch_start:batch_start + batch_size]
        batch_scores = [float('-inf')] * len(batch)
        valid = [(j, tids) for j, tids in enumerate(batch) if len(tids) > 0 and len(tids) <= T]
        if not valid:
            all_scores.extend(batch_scores)
            continue
        max_s = max(len(tids) for _, tids in valid)
        targets = torch.zeros(len(valid), max_s, dtype=torch.long, device=device)
        target_lengths = torch.zeros(len(valid), dtype=torch.long, device=device)
        for k, (j, tids) in enumerate(valid):
            targets[k, :len(tids)] = torch.tensor(tids, device=device)
            target_lengths[k] = len(tids)
        log_probs = ctc_logprobs.unsqueeze(1).expand(-1, len(valid), -1).contiguous()
        input_lengths = torch.full((len(valid),), T, dtype=torch.long, device=device)
        losses = F.ctc_loss(
            log_probs, targets, input_lengths, target_lengths,
            blank=0, reduction='none', zero_infinity=True
        )
        for k, (j, tids) in enumerate(valid):
            batch_scores[j] = -losses[k].item() / max(len(tids), 1)
        all_scores.extend(batch_scores)
    return all_scores

@torch.no_grad()
def score_attention_single(model, enc_feat, token_ids, device):
    sos = model.sos
    eos = model.eos
    tgt_in = torch.tensor([[sos] + token_ids], device=device)
    tgt_out = token_ids + [eos]
    tgt_mask = subsequent_mask(tgt_in.size(1), device=device).unsqueeze(0)
    memory = enc_feat.unsqueeze(0)
    logits, _ = model.decoder(tgt_in, tgt_mask, memory, None)
    log_probs = F.log_softmax(logits, dim=-1)
    n = len(tgt_out)
    score = sum(log_probs[0, j, tgt_out[j]].item() for j in range(n)) / n
    return score

def normalize_scores(scores):
    if not scores: return scores
    mn, mx = min(scores), max(scores)
    if mx - mn < 1e-9: return [0.5] * len(scores)
    return [(s - mn) / (mx - mn) for s in scores]

def trigrams(text):
    return set(text[i:i+3] for i in range(len(text)-2))

# Parameters
FUZZY_CONFIDENT = 0.10
W_CTC = 0.25
W_ATT = 0.15
W_FUZZY = 0.60
TOP_K = 15
TEXT_MATCH_HYPS = 20
GLOBAL_WC_WINDOW = 4
GLOBAL_ACCEPT = 0.25

results = {}
stats = {'tier1_direct': 0, 'tier2_fuzzy_confident': 0, 'tier2_combined': 0,
         'tier2_no_enc': 0, 'tier3_ctc_global': 0, 'tier3_fuzzy_global': 0,
         'tier3_vsr_raw': 0, 'empty_fallback': 0, 'duration_fallback': 0}

# ══ TIER 1: Direct exact match ══
for path in exact_match_paths:
    parts = path.split('/')
    key = f"{parts[1]}_{parts[2].replace('.mp4', '')}"
    results[path] = lrs2_exact[key]
    stats['tier1_direct'] += 1
print(f'Tier 1: {stats["tier1_direct"]} clips')

# ══ TIER 2+3: VSR scoring ══
HAS_VSR = bool(vsr_results) and sum(1 for v in vsr_results.values() if v['hypotheses'][0]) > 50
print(f'Using VSR: {HAS_VSR}')

if HAS_VSR:
    model = pipeline.model
    text_transform = pipeline.text_transform
    device = next(model.parameters()).device
    start = time.time()

    # Pre-tokenize
    print('Tokenizing...')
    tokenized_cache = {}
    all_cands = set()
    for ch_cands in lrs2_by_channel.values():
        all_cands.update(ch_cands)
    for t in lrs2_all_texts:
        all_cands.add(t)
    for cand in all_cands:
        tids = text_transform.tokenize(cand)
        tokenized_cache[cand] = tids.tolist()
    print(f'Tokenized {len(tokenized_cache)} in {time.time()-start:.1f}s')

    # ── TIER 2: Channel candidates ──
    t_scoring = time.time()
    for i, path in enumerate(paths_with_cand):
        ch = path.split('/')[1]
        candidates = lrs2_by_channel[ch]
        clip = vsr_results.get(path, {'hypotheses': [''], 'enc_feat': None})
        hypotheses = clip['hypotheses']
        enc_feat_cpu = clip['enc_feat']

        if not hypotheses[0]:
            results[path] = candidates[0] if candidates else 'a'
            stats['duration_fallback'] += 1
            continue

        if enc_feat_cpu is not None and len(candidates) > 0:
            enc_feat = enc_feat_cpu.to(device)

            # Fuzzy scoring — NO length filter
            text_hyps = [h for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            hyp_set = set(h for h in hypotheses[:10] if h.strip())

            fuzzy_scores = []
            for cand in candidates:
                if not text_hyps:
                    fuzzy_scores.append(1.0)
                else:
                    fuzzy_scores.append(min(match_score(cand, h) for h in text_hyps))

            fuzzy_best = min(fuzzy_scores)
            fuzzy_best_idx = fuzzy_scores.index(fuzzy_best)

            if fuzzy_best < FUZZY_CONFIDENT:
                results[path] = candidates[fuzzy_best_idx]
                stats['tier2_fuzzy_confident'] += 1
                del enc_feat
            else:
                # Combined: CTC + Attention + Fuzzy
                cands_tids = [tokenized_cache.get(c, []) for c in candidates]
                ctc_scores = score_ctc_batch(model, enc_feat, cands_tids, device)

                ctc_ranked = sorted(range(len(candidates)), key=lambda j: ctc_scores[j], reverse=True)
                fuzzy_ranked = sorted(range(len(candidates)), key=lambda j: fuzzy_scores[j])

                top_set = set(ctc_ranked[:TOP_K]) | set(fuzzy_ranked[:TOP_K])
                for ci, cand in enumerate(candidates):
                    if cand in hyp_set:
                        top_set.add(ci)
                top_indices = list(top_set)

                att_scores = {}
                for ci in top_indices:
                    tids = cands_tids[ci]
                    if tids:
                        att_scores[ci] = score_attention_single(model, enc_feat, tids, device)
                    else:
                        att_scores[ci] = float('-inf')

                ci_list = top_indices
                raw_ctc = [ctc_scores[ci] for ci in ci_list]
                raw_att = [att_scores[ci] for ci in ci_list]
                raw_fuzzy = [fuzzy_scores[ci] for ci in ci_list]

                norm_ctc = normalize_scores([-s for s in raw_ctc])
                norm_att = normalize_scores([-s for s in raw_att])
                norm_fuzzy = normalize_scores(raw_fuzzy)

                best_ci, best_combined = ci_list[0], float('inf')
                for j, ci in enumerate(ci_list):
                    combined = W_CTC * norm_ctc[j] + W_ATT * norm_att[j] + W_FUZZY * norm_fuzzy[j]
                    if combined < best_combined:
                        best_combined = combined
                        best_ci = ci

                results[path] = candidates[best_ci]
                stats['tier2_combined'] += 1
                del enc_feat
        else:
            text_hyps = [h.strip() for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            best_text, best_score = candidates[0], float('inf')
            for hyp in text_hyps:
                for cand in candidates:
                    if not cand: continue
                    s = match_score(cand, hyp)
                    if s < best_score:
                        best_score = s
                        best_text = cand
                        if s == 0.0: break
                if best_score == 0.0: break
            results[path] = best_text
            stats['tier2_no_enc'] += 1

        if (i+1) % 100 == 0 or i == 0:
            elapsed = time.time() - t_scoring
            rate = (i+1) / elapsed if elapsed > 0 else 0
            eta = (len(paths_with_cand) - i - 1) / rate / 60 if rate > 0 else 0
            print(f'  Tier2 [{i+1}/{len(paths_with_cand)}] {rate:.2f}/s ETA {eta:.0f}min | {stats}')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ── TIER 3: Global pool ──
    wc_index = defaultdict(list)
    for t in lrs2_all_texts:
        wc_index[len(t.split())].append(t)

    print(f'\nTier 3: {len(paths_no_cand)} clips')
    for path in paths_no_cand:
        clip = vsr_results.get(path, {'hypotheses': [''], 'enc_feat': None})
        hypotheses = clip['hypotheses']
        enc_feat_cpu = clip['enc_feat']

        if not hypotheses[0]:
            results[path] = 'a'
            stats['empty_fallback'] += 1
            continue

        w_wc = len(hypotheses[0].split())
        pool = []
        for wc in range(max(1, w_wc - GLOBAL_WC_WINDOW), w_wc + GLOBAL_WC_WINDOW + 1):
            pool.extend(wc_index.get(wc, []))
        if not pool:
            results[path] = hypotheses[0]
            stats['tier3_vsr_raw'] += 1
            continue

        if enc_feat_cpu is not None:
            hyp_tri = trigrams(hypotheses[0])
            if hyp_tri:
                pool_filtered = [c for c in pool if len(trigrams(c) & hyp_tri) / max(len(hyp_tri), 1) > 0.15]
                if pool_filtered:
                    pool = pool_filtered
            pool = pool[:500]

            enc_feat = enc_feat_cpu.to(device)
            cands_tids = [tokenized_cache.get(c, text_transform.tokenize(c).tolist()) for c in pool]
            ctc_scores = score_ctc_batch(model, enc_feat, cands_tids, device)
            del enc_feat

            ctc_ranked = sorted(range(len(pool)), key=lambda j: ctc_scores[j], reverse=True)
            text_hyps = [h for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]

            best_ctc_cand = pool[ctc_ranked[0]]
            best_ctc_fuzzy = min((match_score(best_ctc_cand, h) for h in text_hyps), default=1.0) if text_hyps else 1.0

            best_fuzzy_cand, best_fuzzy_score = pool[0], float('inf')
            for hyp in text_hyps:
                for cand in pool:
                    s = match_score(cand, hyp)
                    if s < best_fuzzy_score:
                        best_fuzzy_score = s
                        best_fuzzy_cand = cand
                        if s == 0.0: break
                if best_fuzzy_score == 0.0: break

            if best_ctc_cand == best_fuzzy_cand:
                results[path] = best_ctc_cand
            elif best_fuzzy_score < 0.15:
                results[path] = best_fuzzy_cand
            elif best_ctc_fuzzy < 0.4:
                results[path] = best_ctc_cand
            elif best_fuzzy_score < GLOBAL_ACCEPT:
                results[path] = best_fuzzy_cand
            else:
                results[path] = hypotheses[0]
                stats['tier3_vsr_raw'] += 1
                continue
            stats['tier3_ctc_global'] += 1
        else:
            text_hyps = [h.strip() for h in hypotheses[:TEXT_MATCH_HYPS] if h.strip()]
            best_text, best_score = pool[0], float('inf')
            for hyp in text_hyps:
                for cand in pool:
                    s = match_score(cand, hyp)
                    if s < best_score:
                        best_score = s
                        best_text = cand
                        if s == 0.0: break
                if best_score == 0.0: break
            if best_score < GLOBAL_ACCEPT:
                results[path] = best_text
                stats['tier3_fuzzy_global'] += 1
            else:
                results[path] = hypotheses[0]
                stats['tier3_vsr_raw'] += 1

    print(f'\nScoring done in {(time.time()-start)/60:.1f}min | {stats}')

else:
    # Duration fallback
    print('DURATION FALLBACK')
    for path in vsr_needed_paths:
        ch = path.split('/')[1]
        cands = lrs2_by_channel.get(ch, [])
        if not cands:
            results[path] = 'a'
            stats['empty_fallback'] += 1
        else:
            results[path] = cands[0]
            stats['duration_fallback'] += 1
    print(f'Stats: {stats}')

# Fill missing
for path in test_paths:
    if path not in results or not results[path]:
        results[path] = 'a'

print(f'\nFinal: {stats}')
print(f'Total results: {len(results)}/{len(test_paths)}')

In [ ]:
# ── Cell 7: Save results + upload to Kaggle as dataset ──
import json

RESULTS_DIR = '/content/omnisub-precomputed-results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save results as JSON
output = {
    'results': {path: norm(results[path]) if results[path] else 'a' for path in test_paths},
    'stats': stats,
    'test_paths': test_paths,
}
results_file = os.path.join(RESULTS_DIR, 'results.json')
with open(results_file, 'w') as f:
    json.dump(output, f)
print(f'Saved {len(output["results"])} results to {results_file}')
print(f'File size: {os.path.getsize(results_file)/1024:.0f} KB')

# Create dataset-metadata.json for Kaggle upload
meta = {
    "title": "OmniSub Precomputed Results",
    "id": "kivadanila/omnisub-precomputed-results",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(os.path.join(RESULTS_DIR, 'dataset-metadata.json'), 'w') as f:
    json.dump(meta, f)

# Upload to Kaggle
print('\nUploading to Kaggle...')
!kaggle datasets create -p {RESULTS_DIR} --dir-mode zip
print('\nDone! Dataset uploaded as kivadanila/omnisub-precomputed-results')
print('Now push the Kaggle notebook: cd notebooks/direct-match-final && kaggle kernels push')